# H5 見た目で予想する：Gemini（VLM）

**ねらい**：作品の 0 番目の工程。カメラの写真を **VLM**（画像と言葉を両方扱う AI。ここでは Gemini Robotics-ER）に見せ、箱ごとに
**材質・重さ・壊れやすさ・自信** を JSON で答えさせる。その答えで、その 1 の **試し掴みの探し方**（H3：弱い順／強めから）を選ぶ。
- 指示文を書き換えると、答えがどう変わるか
- **自信** を使わずに答えを信じると、何が起きるか（わざと失敗 `trust_ai`）

このノートは Genesis を使わないので、すぐ動く。

## 1. 準備
Gemini のキーは、Colab の左の 🔑（シークレット）に、名前 `GEMINI_API_KEY` で入れて、「ノートブックからのアクセス」をオンにする。
**キーがなくても進められる**（作品で保存した答えを使う）。

In [ ]:
import json, base64, io, os
if "google.colab" in str(get_ipython()):
    !pip -q install google-genai==2.29.0
import numpy as np, matplotlib.pyplot as plt
try:
    from google.colab import userdata
    KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    KEY = os.environ.get("GEMINI_API_KEY")
print("キー：", "あり（その場で Gemini を呼ぶ）" if KEY else "なし（保存した答えを使う）")

## 2. 写真を見る
作品のシミュレータで撮った写真。左から、段ボール（30 g）・発泡スチロール（10 g）・段ボール（150 g）・金属（400 g）・段ボール（300 g）。
**段ボールの 3 つは、見た目がまったく同じ**（中身の重さだけ違う）。

In [ ]:
PNG_B64 = "..."
open("boxes.png", "wb").write(base64.b64decode(PNG_B64))
from PIL import Image
img = Image.open("boxes.png"); print(img.size)
plt.figure(figsize=(10, 4)); plt.imshow(img); plt.axis("off"); plt.show()
TRUTH = [("段ボール", 30, True), ("発泡スチロール", 10, True), ("段ボール", 150, True), ("金属", 400, False), ("段ボール", 300, True)]   # 左から（材質, 重さ g, 壊れやすい）

## 3. Gemini に聞く
指示文（英語）は作品と同じ。`point` は箱の中心の位置（0〜1000 に正規化した [y, x]）。

In [ ]:
PROMPT = """This image shows small boxes (each about 3 cm) on a table, seen by a robot that must pick them up.
For EVERY box, return one JSON object:
- "point": [y, x] of the box center, normalized to 0-1000
- "looks_like": what the box looks like it is made of
- "weight_g": your best guess of its total weight in grams (contents included), as [low, high]
- "fragile": true if squeezing it hard could crush or damage it
- "confidence": "low", "medium" or "high" for the weight guess. If boxes look identical, say so and give a wide range.
Return only a JSON array, no prose."""
SAVED = '```json\n[\n  {\n    "point": [481, 180],\n    "looks_like": "cardboard",\n    "weight_g": [2, 50],\n    "fragile": true,\n    "confidence": "low"\n  },\n  {\n    "point": [481, 338],\n    "looks_like": "stone",\n    "weight_g": [50, 100],\n    "fragile": false,\n    "confidence": "medium"\n  },\n  {\n    "point": [481, 500],\n    "looks_like": "cardboard",\n    "weight_g": [2, 50],\n    "fragile": true,\n    "confidence": "low"\n  },\n  {\n    "point": [481, 661],\n    "looks_like": "metal",\n    "weight_g": [150, 250],\n    "fragile": false,\n    "confidence": "high"\n  },\n  {\n    "point": [481, 820],\n    "looks_like": "cardboard",\n    "weight_g": [2, 50],\n    "fragile": true,\n    "confidence": "low"\n  }\n]\n```'

def ask(prompt=PROMPT, model="gemini-robotics-er-2-preview"):
    """写真と指示文を Gemini に渡し、答えの文字列を返す。キーがなければ、作品で保存した答え"""
    if not KEY:
        return SAVED
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=KEY, http_options=types.HttpOptions(timeout=120_000, retry_options=types.HttpRetryOptions(attempts=1)))
    f = client.files.upload(file="boxes.png")
    resp = client.interactions.create(model=model,
                                      input=[{"type": "image", "uri": f.uri, "mime_type": f.mime_type}, {"type": "text", "text": prompt}],
                                      generation_config={"thinking_level": "medium"})
    return resp.output_text

def parse(text):
    body = text.split("```json")[-1].split("```")[0] if "```" in text else text
    return sorted(json.loads(body), key=lambda a: a["point"][1])      # 左から順に

def compare(answers):
    print(f"{'#':2s} {'本当':16s} {'Gemini の答え':40s}")
    for i, (a, (mat, g, frag)) in enumerate(zip(answers, TRUTH)):
        print(f"{i:2d} {mat}・{g} g・{'壊れやすい' if frag else '壊れにくい':5s}   {a['looks_like']}・{a['weight_g']} g・fragile={a['fragile']}・自信 {a['confidence']}")

text = ask(); print(text[:400], "..."); answers = parse(text); compare(answers)

**期待される出力**（作品で保存した答え）
- 段ボールの 3 つは、同じ答え（`cardboard`、2〜50 g、壊れやすい、自信 low）。**中身の重さは見た目では分からない** と自分で言っている
- 金属は `metal`、150〜250 g、自信 high（本当は 400 g。半分近くに見積もった）
- 発泡スチロールを **`stone`（石）、壊れにくい、自信 medium** と見誤る

キーがある人は、その場で呼んだ答えが、保存した答えと同じとは限らない（VLM の答えは毎回少し揺れる）。
**2026-10-22 に手元でその場で呼んだら、発泡スチロールを「石・壊れにくい・自信 high」と答えた**（保存した答えは medium）。4 で、これが何を意味するかを見る。

## ✏️ 変えてみる 1：指示文（キーがある人）
**予想**：最後の「If boxes look identical, say so and give a wide range.」を消すと、段ボールの 3 つの答えはどうなる？

In [ ]:
if KEY:
    p2 = PROMPT.replace(" If boxes look identical, say so and give a wide range.", "")
    compare(parse(ask(p2)))
else:
    print("キーがないので飛ばす")

**期待される出力**（2026-10-22 に手元で 1 回呼んだ結果）：段ボールの 3 つが「5〜25 g・自信 medium」になった（消す前は 2〜100 g・low）。
「見た目が同じなら幅を広く」と言わないと、**根拠がないのに、幅を狭くして自信を上げる**。指示文の 1 文が、答えの慎重さを決めている。

## 4. 答えで探し方を選ぶ
作品の決まり：**「壊れにくい」かつ「自信 high」の箱だけ、強めから試す**（QUICK）。それ以外は弱い順（GENTLE）。
H3 で確かめたとおり、強めから試すと、挟む力は 11 N。段ボール（15 N）は無事でも、**発泡スチロール（6 N）は潰れる**。

In [ ]:
STRENGTH = {"段ボール": 15.0, "発泡スチロール": 6.0, "金属": float("inf")}   # 箱の強さ（挟む力の上限 N）
PEAK = {"GENTLE": 4.3, "QUICK": 10.9}                                         # H3 で測った、挟む力の最大 [N]

def choose(a, trust_ai=False):
    sturdy = (not a.get("fragile", True)) and (a.get("confidence") == "high" or trust_ai)
    return "QUICK" if sturdy else "GENTLE"

def plan(answers, trust_ai=False, title=None):
    print(title or ("わざと失敗 trust_ai（自信によらず信じる）" if trust_ai else "作品の決まり（自信 high のときだけ信じる）"))
    for a, (mat, g, _) in zip(answers, TRUTH):
        way = choose(a, trust_ai)
        crushed = PEAK[way] > STRENGTH[mat]
        print(f"   {mat:8s} {g:4d} g   答え {a['looks_like']:10s}（自信 {a['confidence']:6s}） → {way:6s}   挟む力の最大 {PEAK[way]:4.1f} N   {'潰す' if crushed else '無事'}")

answers = parse(SAVED)          # ここからは、保存した答えで比べる（毎回同じ結果になるように）
plan(answers)
plan(answers, trust_ai=True)

**期待される出力**：作品の決まりでは、金属だけ QUICK、ほかは GENTLE で、5 つとも無事。
`trust_ai` では、発泡スチロール（`stone`、自信 medium）も QUICK になり、**潰す**（動画の場面 5 の失敗）。

**注意**：その場で呼んで「石・自信 high」と返った答え（3 の注）を使うと、**作品の決まりでも潰す**。自信で守る決まりは、答えの揺れで破られることがある。
キーがある人は、`answers = parse(text)`（その場で呼んだ答え）にして `plan(answers)` を ▶ し直して確かめる。

## ✏️ 変えてみる 2：信じる自信の境目
**予想**：`"high"` だけでなく `"medium"` も信じる決まりにすると？ 逆に、誰も信じない（全部 GENTLE）と、何を失う？

In [ ]:
def choose(a, trust_ai=False):
    sturdy = (not a.get("fragile", True)) and (a.get("confidence") in ("high", "medium") or trust_ai)   # ← ここを書き換える
    return "QUICK" if sturdy else "GENTLE"
plan(answers, title="書き換えた決まり")

**期待される出力**：medium も信じると、`trust_ai` と同じく発泡スチロールを潰す。全部 GENTLE なら潰さないが、金属（400 g）も 0.2 から順に試すので、試す回数が増える（作品の通しでは、金属は QUICK の 1 回目で持てた）。

## 確かめる問い
1. 段ボールの 3 つで、Gemini が同じ答えを返したのはなぜ？ 作品はそれをどう補っている？
2. 自信が medium の答えを信じないのはなぜ？
3. VLM の答えをロボットの動きに使うとき、気をつけることは？

<details><summary>答え</summary>

1. 見た目が同じで、中身の重さは写真に写らない。作品は、持って確かめる（試し掴みで力加減、肩の力で重さ）
2. 見誤り（発泡スチロールを石）は、自信が中くらいの答えに出た。信じて強めから試すと、壊れやすい箱を潰す。外れたときに取り返しがつかない判断にだけ、高い自信を求める
3. 答えは毎回揺れ、見誤ったまま自信 high と言うこともある（2026-10-22 の発泡スチロール）。自信の決まりだけに頼らず、強めから試すときも、挟む力の見張り（潰れの手前で止める）と、持って確かめる手順を組み合わせる
</details>